# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane:** content-refresh triage (same as Week 4). **Target:** `is_declining_label`
(1 = `trend_direction == "down"`) — a yes/no outcome that was actually observed, not a proxy.

Per the toolkit's own guidance: *"yes/no with an observed label → Logistic Regression, then
Random Forest."* That's the path here — a simple, readable Logistic Regression first, then a
Random Forest to see how much a stronger, non-linear method actually buys over it. Simplicity
is treated as a feature, not a starting handicap: if the Random Forest doesn't clearly beat
Logistic Regression on the comparison table in Section 3, that's a real finding, not a failure
to report.

### Feature selection — deliberately leakage-free

Excluded on purpose (see code comments below for the full reasoning):
- `content_id`, `client_id` — pseudonyms, used only for grouping/joins, never as features
- `trend_pct`, `trend_direction`, `is_declining_label` — the label and its direct source
- `impressions_last_30d` / `_prev_30d` and the matching clicks/sessions columns — these are
  literally what `trend_pct` and `trend_direction` are computed *from*. Including them would
  let a model reconstruct the label almost exactly — the same leakage as using `trend_pct`
  directly, one step removed.
- `provider_used`, `model_used` — the data dictionary marks these "not a model feature"

Everything else — the 90-day totals, the derived rates, the keyword-context columns, and the
transparent tier/bucket columns — is fair game and gets used.

### Reproducibility

`RANDOM_STATE = 42`, fixed everywhere a model or split touches randomness, so re-running this
notebook reproduces the same table in Section 3.


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option("display.width", 120)

# --- make sure we're inside the repo (safe to re-run: won't re-clone if already present) ---
if not os.path.exists("flyrank-ml-internship-starter"):
    get_ipython().system('git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git')
if os.path.basename(os.getcwd()) != "flyrank-ml-internship-starter":
    os.chdir("flyrank-ml-internship-starter")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"loaded {len(df):,} rows x {df.shape[1]} columns")

# target isn't in the raw 44-col CSV - derived here using the documented rule
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"base rate (whole dataset): {df['is_declining_label'].mean():.3f}")

# ------------------------------------------------------------------
# Feature lists - see Section 1 markdown for what's excluded and why.
# ------------------------------------------------------------------
numeric_features = [
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "search_volume", "competition", "cpc",
]
categorical_features = [
    "content_type", "main_intent", "competition_level",
    "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier",
]

# missingness follows content_type (per the data dictionary) - flag it before filling,
# so a blind fillna(0) doesn't quietly encode "this content_type has no keyword data"
# as if it were a genuine zero.
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
numeric_features += ["has_keyword_data", "has_word_count"]

X_numeric = df[numeric_features].fillna(0)
X_categorical = pd.get_dummies(df[categorical_features].fillna("unknown"), prefix=categorical_features)

X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_declining_label"]
groups = df["client_id"]

print(f"feature matrix: {X.shape[0]:,} rows x {X.shape[1]} columns")


## 2. Split design

**Grouped by `client_id`**, not a plain random row split, and not time-aware.

Why grouped: this CSV is a single trailing-90-day snapshot — there's no repeated time axis to
split on here (that's what the warehouse release's `fact_content_daily_performance` is for, in
a later week). But pages from the same client share client-level baseline traffic, template,
and content strategy. A random row split would let pages from the same client land in both
train and test, so the model could partly learn "this is client X's style" rather than the
general pattern — an easier, less honest problem than the one it'll face on a brand-new client.
`GroupShuffleSplit` on `client_id` keeps every client's pages entirely on one side of the split,
which is the same reasoning the `flyrank-data` skill gives for grouped train/test splits.

80/20 split, one fold (a single honest split is enough at this stage), `random_state=42` for
reproducibility.


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

overlap = set(groups_train) & set(groups_test)
print(f"train: {len(X_train):,} rows, {groups_train.nunique()} clients")
print(f"test:  {len(X_test):,} rows, {groups_test.nunique()} clients")
print(f"client overlap between train/test: {len(overlap)} (should be 0)")
print(f"train base rate: {y_train.mean():.3f}  |  test base rate: {y_test.mean():.3f}")


## 3. Train + compare vs my baseline

Same test split, same three metrics (precision@20/50/200), for all three: the Week-4 rule
baseline (`stale_but_visible`), Logistic Regression, and Random Forest. The baseline is
**recomputed on the test rows only** — scoring it on the same slice the models are judged on is
what makes the comparison fair; scoring it on the full dataset (like Week 4 did) would let it
"see" rows the models never get credit for.

Logistic Regression runs on standardized features (via a `Pipeline`, fit only on train). Random
Forest runs on the raw feature matrix — trees don't need scaling. Both use `random_state=42`.


In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- Week-4 baseline, recomputed on the TEST split only ---
STALE_DAYS_THRESHOLD = 180
VISIBLE_IMPRESSIONS_THRESHOLD = 500

baseline_test = df.loc[test_idx].copy()
stale = (baseline_test["days_since_last_update"] >= STALE_DAYS_THRESHOLD).astype(int)
visible = (baseline_test["impressions_90d"] >= VISIBLE_IMPRESSIONS_THRESHOLD).astype(int)
baseline_scores = (stale * visible * baseline_test["impressions_90d"]).values

# --- Logistic Regression ---
log_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
log_reg.fit(X_train, y_train)
lr_scores = log_reg.predict_proba(X_test)[:, 1]

# --- Random Forest ---
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

# --- comparison table: same split, same metrics, base rate included ---
rows = []
for name, scores in [
    ("baseline (Week 4 rule)", baseline_scores),
    ("Logistic Regression", lr_scores),
    ("Random Forest", rf_scores),
]:
    row = {"model": name, "roc_auc": roc_auc_score(y_test, scores)}
    for k in (20, 50, 200):
        row[f"precision@{k}"] = precision_at_k(scores, y_test.values, k)
    rows.append(row)

comparison = pd.DataFrame(rows).set_index("model")
comparison.insert(0, "base_rate(test)", y_test.mean())
print(comparison.round(3))


## 4. Errors and interpretation

Two importance views on the Random Forest (built-in `feature_importances_`, then permutation
importance on the held-out test set) — printed side by side so a suspiciously perfect top
feature in one is checkable against the other rather than trusted on its own. Then: error rate
broken out by `content_type` and `freshness_tier`, and three concrete false negatives / false
positives with the numbers behind each.

**Fill in after running:** once the cell below prints real numbers, replace this line with 2-3
sentences — which model won the comparison table above, at which K, by how much; what the top
features actually are and whether they plausibly relate to decline (or look leakage-suspicious);
and what the worst error group has in common. That reading is the part a template can't do for
you — the assignment asks you to validate the result yourself.


In [ ]:
# --- feature importance: built-in vs permutation, checked against each other ---
rf_importance = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Random Forest - top 10 features (built-in importance):")
print(rf_importance.head(10))

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
print("\nRandom Forest - top 10 features (permutation importance, test set):")
print(perm_importance.head(10))

# --- where is the model most wrong? ---
test_df = df.loc[test_idx].copy()
test_df["y_true"] = y_test.values
test_df["rf_score"] = rf_scores
test_df["rf_pred"] = (rf_scores >= 0.5).astype(int)
test_df["is_error"] = test_df["y_true"] != test_df["rf_pred"]

n_errors = test_df["is_error"].sum()
print(f"\n{n_errors} of {len(test_df)} test rows misclassified at a 0.5 threshold ({n_errors/len(test_df):.1%})")

print("\nerror rate by content_type:")
print(test_df.groupby("content_type")["is_error"].agg(n="size", error_rate="mean").round(3))

print("\nerror rate by freshness_tier:")
print(test_df.groupby("freshness_tier")["is_error"].agg(n="size", error_rate="mean").round(3))

# --- 3 concrete wrong cases each way ---
false_negatives = test_df[(test_df["y_true"] == 1) & (test_df["rf_pred"] == 0)].nlargest(3, "impressions_90d")
false_positives = test_df[(test_df["y_true"] == 0) & (test_df["rf_pred"] == 1)].nlargest(3, "rf_score")

cols = ["content_id", "content_type", "days_since_last_update", "impressions_90d", "avg_position", "rf_score"]
print("\n3 false negatives (declining pages the model missed - highest-visibility ones):")
print(false_negatives[cols])

print("\n3 false positives (model flagged as declining, actually flat/up):")
print(false_positives[cols])


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.